In [1]:
import sklearn
import pandas as pd

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.neural_network import MLPClassifier
import joblib
import numpy as np

sklearn.set_config(transform_output="pandas")
pd.set_option('display.max_colwidth', None)

model = MLPClassifier(
    early_stopping=True,
    solver='adam',
    activation='relu',
    max_iter=300,
    hidden_layer_sizes=(128, 64, 32),
    batch_size=32
)

transformer = joblib.load("transformer.pkl")
train_df    = pd.read_csv("data/balanced_train_data.csv")

y        = train_df['TARGET'].astype(np.float32)
train_df = train_df.drop(columns=['TARGET'])

X_transformed = transformer.transform(train_df).astype(np.float32)

del train_df

sample_weights = compute_sample_weight(class_weight='balanced', y=y)
model.fit(X_transformed, y, sample_weight=sample_weights)

test_df = pd.read_csv("data/split_data_test.csv")
y_test = test_df['TARGET'].astype(np.float32)
X_test = test_df.drop(columns=['TARGET'])
transformed_test_df = transformer.transform(X_test).astype(np.float32)

del X_test

y_probs = model.predict_proba(X=transformed_test_df)[:, 1]
y_pred  = model.predict(X=transformed_test_df)

result = pd.DataFrame({
    'library': ["scikit-learn"],
    'algorithm': ['MLPClassifier'],
    'hyerparamters':[str(model.get_params())],
    'accuracy': [accuracy_score(y_test, y_pred)],
    'auc': [roc_auc_score(y_test, y_probs)]
}, index=None)

print(result.T)

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   0
library                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 scikit-lear